# Mask Re-labelling

Some patches have acceptable echogram signal but bad masks (erratic bottom
lines, no physical boundary underneath, large jumps). Rather than excluding
these patches, this notebook tries to improve their masks by re-running
bottom detection with multiple methods and keeping the best result.

The `relabel_patch` function works at the patch level by reversing the
[0,1] normalisation to approximate dB values, running TVG, DP, and
morphological detectors, and scoring each candidate. If no method beats
the original mask, it stays unchanged. Targeted fixes (median smoothing
for spiky bottoms, interpolation for gaps) are also available.

In [ ]:
import sys
sys.path.insert(0, '../..')

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

from data_cleaning import load_patch_by_uid, load_quality_df, MASK_METRIC_COLS
from data_cleaning.scoring import compute_mask_quality_score, compute_combined_score
from data_cleaning.visualization import plot_patch_with_mask
from data_cleaning.relabel import (
    relabel_patch, smooth_erratic_bottom, fill_bottom_gaps,
)
from data_cleaning.metrics import compute_mask_metrics

PATCH_DIR = Path('/Users/axelhagen/Master/Datasets/patch_cache')
COMBINED_CSV = Path('/Users/axelhagen/Master/Datasets/combined_quality.csv')
RELABEL_DIR = Path('/Users/axelhagen/Master/Datasets/patch_cache_relabelled')
RELABEL_DIR.mkdir(exist_ok=True)

## Identify candidates for re-labelling

Patches with bad mask scores but acceptable signal quality are the best
candidates — the echogram contains a visible bottom, but the mask didn't
capture it well.

In [ ]:
df = pd.read_csv(COMBINED_CSV)

# Need scores — recompute if not in CSV
if 'mask_quality_score' not in df.columns:
    df['mask_quality_score'] = compute_mask_quality_score(df)
if 'patch_quality_score' not in df.columns:
    from data_cleaning.scoring import compute_patch_quality_score
    df['patch_quality_score'] = compute_patch_quality_score(df)

# Candidates: bad mask, decent signal
MASK_THRESHOLD = 0.30
SIGNAL_THRESHOLD = 0.40

candidates = df[
    (df['mask_quality_score'] < MASK_THRESHOLD) &
    (df['patch_quality_score'] >= SIGNAL_THRESHOLD)
].sort_values('mask_quality_score')

print(f'Total patches: {len(df)}')
print(f'Bad mask + decent signal: {len(candidates)}')
print(f'Mask score range: [{candidates["mask_quality_score"].min():.3f}, '
      f'{candidates["mask_quality_score"].max():.3f}]')

## Preview: worst candidates before re-labelling

In [ ]:
preview_uids = candidates.head(12)['uid'].tolist()

fig, axes = plt.subplots(3, 4, figsize=(16, 9))
for ax, uid in zip(axes.flat, preview_uids):
    patch, mask = load_patch_by_uid(uid, PATCH_DIR)
    plot_patch_with_mask(patch, mask, uid, ax)
for ax in axes.flat[len(preview_uids):]:
    ax.set_visible(False)
fig.suptitle('Candidates for re-labelling (original masks)', fontweight='bold')
plt.tight_layout()
plt.show()

## Run re-labelling pipeline

For each candidate, try all detection methods and keep the best. Also
apply targeted fixes (smoothing and gap filling) as a separate pass.

In [ ]:
results = []

for _, row in tqdm(candidates.iterrows(), total=len(candidates), desc='Re-labelling'):
    uid = row['uid']
    patch, mask = load_patch_by_uid(uid, PATCH_DIR)
    
    # Try re-detection with multiple methods
    new_mask, method, improvement = relabel_patch(patch, mask)
    
    # Also try targeted fixes on the original mask
    smoothed = smooth_erratic_bottom(mask)
    filled = fill_bottom_gaps(smoothed)
    
    # Compare: relabelled vs fixed original
    from data_cleaning.relabel import _score_mask
    fix_score = _score_mask(patch, filled)
    relabel_score = _score_mask(patch, new_mask)
    original_score = _score_mask(patch, mask)
    
    # Pick the best
    if relabel_score >= fix_score:
        best_mask = new_mask
        best_method = method
        best_score = relabel_score
    else:
        best_mask = filled
        best_method = 'smooth+fill'
        best_score = fix_score
    
    results.append({
        'uid': uid,
        'original_score': original_score,
        'best_score': best_score,
        'improvement': best_score - original_score,
        'method': best_method,
        'mask': best_mask,
    })

results_df = pd.DataFrame([{k: v for k, v in r.items() if k != 'mask'} for r in results])
print(f'\nResults:')
print(f'  Improved: {(results_df["improvement"] > 0).sum()}')
print(f'  Unchanged: {(results_df["improvement"] <= 0).sum()}')
print(f'  Methods used: {results_df["method"].value_counts().to_dict()}')

## Before/after comparison

In [ ]:
# Show the patches with the biggest improvements
improved = results_df[results_df['improvement'] > 0].nlargest(8, 'improvement')
improved_results = [r for r in results if r['uid'] in improved['uid'].values]

if improved_results:
    fig, axes = plt.subplots(len(improved_results), 2, 
                             figsize=(10, 3 * len(improved_results)))
    if len(improved_results) == 1:
        axes = axes[np.newaxis, :]
    
    for row_idx, r in enumerate(improved_results):
        patch, orig_mask = load_patch_by_uid(r['uid'], PATCH_DIR)
        
        plot_patch_with_mask(patch, orig_mask, 
                           f'{r["uid"]}\noriginal (score={r["original_score"]:.3f})',
                           axes[row_idx, 0])
        plot_patch_with_mask(patch, r['mask'],
                           f'{r["uid"]}\n{r["method"]} (score={r["best_score"]:.3f})',
                           axes[row_idx, 1])
    
    fig.suptitle('Biggest improvements: original (left) vs relabelled (right)',
                 fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print('No patches were improved by re-labelling.')

## Recompute quality metrics for improved patches

In [ ]:
improved_results_all = [r for r in results if r['improvement'] > 0]

if improved_results_all:
    new_metrics = []
    for r in improved_results_all:
        patch, _ = load_patch_by_uid(r['uid'], PATCH_DIR)
        m = compute_mask_metrics(patch, r['mask'])
        m['uid'] = r['uid']
        new_metrics.append(m)
    
    new_df = pd.DataFrame(new_metrics)
    
    # Compare distributions
    for col in ['bottom_curvature', 'mask_edge_intensity', 'bottom_snr']:
        if col in new_df.columns:
            orig_vals = candidates.loc[candidates['uid'].isin(new_df['uid']), col]
            new_vals = new_df[col]
            print(f'{col}: before={orig_vals.median():.3f} -> after={new_vals.median():.3f}')
    
    print(f'\n{len(improved_results_all)} patches improved')
else:
    print('No improvements to analyse.')

## Save re-labelled patches

Write improved patches to a separate directory. The original NPZ files are
never modified. The training pipeline can be pointed at the relabelled cache
by updating `cache_dir` in the config.

In [ ]:
improved_results_all = [r for r in results if r['improvement'] > 0]

if not improved_results_all:
    print('No patches to save.')
else:
    # Group improved patches by NPZ file
    from collections import defaultdict
    by_npz = defaultdict(dict)
    for r in improved_results_all:
        stem, idx_str = r['uid'].split('::')
        by_npz[stem][int(idx_str)] = r['mask']
    
    # For each NPZ: load original, replace improved masks, save to RELABEL_DIR
    for stem, idx_mask_map in by_npz.items():
        orig_path = PATCH_DIR / f'{stem}.npz'
        data = np.load(orig_path)
        patches = data['patches'].copy()
        masks = data['masks'].copy()
        
        for idx, new_mask in idx_mask_map.items():
            masks[idx] = new_mask
        
        out_path = RELABEL_DIR / f'{stem}.npz'
        np.savez_compressed(out_path, patches=patches, masks=masks)
        print(f'  {stem}: updated {len(idx_mask_map)} masks -> {out_path.name}')
    
    # Copy unchanged NPZ files so the relabelled dir is a complete replacement
    import shutil
    all_npz = sorted(PATCH_DIR.glob('*.npz'))
    copied = 0
    for npz_path in all_npz:
        dest = RELABEL_DIR / npz_path.name
        if not dest.exists():
            shutil.copy2(npz_path, dest)
            copied += 1
    
    print(f'\nSaved {len(by_npz)} modified NPZ files to {RELABEL_DIR}')
    print(f'Copied {copied} unchanged NPZ files')
    print(f'Total: {len(list(RELABEL_DIR.glob("*.npz")))} NPZ files in relabelled cache')